# [Problem 1] Scratch implementation of BoW

In [1]:
import collections

mini_dataset = [
    "This movie is SOOOO funny!!!!",
    "What a movie!",
    "best movie ever!!!!! this movie",
]

# --- Preprocessing & Tokenization ---
def preprocess(text):
    # Lowercase and remove all non-word characters (including punctuation)
    text = text.lower()
    text = ''.join(c for c in text if c.isalnum() or c.isspace())
    # Split by spaces to get tokens
    return text.split()

tokenized_sentences = [preprocess(s) for s in mini_dataset]
# [['this', 'movie', 'is', 'soooo', 'funny'], ['what', 'a', 'movie'], ['best', 'movie', 'ever', 'this', 'movie']]


# --- 1-GRAM (UNIGRAM) IMPLEMENTATION ---
print("--- 1-GRAM (UNIGRAM) BOw ---")
# 1. Create Vocabulary and Global Counts
vocab_1gram = set()
for tokens in tokenized_sentences:
    vocab_1gram.update(tokens)
vocab_1gram = sorted(list(vocab_1gram)) # For consistent column order
print("Vocabulary (1-gram):", vocab_1gram)

# 2. Calculate BoW Vector for each sentence
bow_vectors_1gram = []
for tokens in tokenized_sentences:
    # Use Counter to get counts for the current sentence
    counts = collections.Counter(tokens)
    
    # Create the vector based on the global vocabulary
    vector = [counts.get(word, 0) for word in vocab_1gram]
    bow_vectors_1gram.append(vector)

# Print the final result (same as the table in section B)
print("BoW Vectors (1-gram):")
for i, vector in enumerate(bow_vectors_1gram):
    print(f"Sentence {i+1}: {vector}")


# --- 2-GRAM (BIGRAM) IMPLEMENTATION ---
print("\n--- 2-GRAM (BIGRAM) BOw ---")
# 1. Create 2-gram Tokens and Global Vocabulary
all_2grams = []
for tokens in tokenized_sentences:
    # Create 2-gram tokens: (word1, word2), (word2, word3), ...
    twograms = [" ".join(tokens[i:i+2]) for i in range(len(tokens) - 1)]
    all_2grams.append(twograms)
    
vocab_2gram = set(g for grams in all_2grams for g in grams)
vocab_2gram = sorted(list(vocab_2gram))
print("Vocabulary (2-gram):", vocab_2gram)

# 2. Calculate BoW Vector for each sentence
bow_vectors_2gram = []
for twograms in all_2grams:
    # Use Counter for the 2-gram tokens in the current sentence
    counts = collections.Counter(twograms)

    # Create the vector based on the global 2-gram vocabulary
    vector = [counts.get(word, 0) for word in vocab_2gram]
    bow_vectors_2gram.append(vector)

# Print the final result (same as the table in section C)
print("BoW Vectors (2-gram):")
for i, vector in enumerate(bow_vectors_2gram):
    print(f"Sentence {i+1}: {vector}")

--- 1-GRAM (UNIGRAM) BOw ---
Vocabulary (1-gram): ['a', 'best', 'ever', 'funny', 'is', 'movie', 'soooo', 'this', 'what']
BoW Vectors (1-gram):
Sentence 1: [0, 0, 0, 1, 1, 1, 1, 1, 0]
Sentence 2: [1, 0, 0, 0, 0, 1, 0, 0, 1]
Sentence 3: [0, 1, 1, 0, 0, 2, 0, 1, 0]

--- 2-GRAM (BIGRAM) BOw ---
Vocabulary (2-gram): ['a movie', 'best movie', 'ever this', 'is soooo', 'movie ever', 'movie is', 'soooo funny', 'this movie', 'what a']
BoW Vectors (2-gram):
Sentence 1: [0, 0, 0, 1, 0, 1, 1, 1, 0]
Sentence 2: [1, 0, 0, 0, 0, 0, 0, 0, 1]
Sentence 3: [0, 1, 1, 0, 1, 0, 0, 1, 0]


In [ ]:
# [Problem 2] TF-IDF calculation

In [ ]:
import nltk
import pandas as pd
from sklearn.datasets import load_files
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords
import numpy as np

# --- 1. Download NLTK Stopwords (Run this once) ---
try:
    # Attempt to load, if fails, download
    stop_words_list = stopwords.words("english")
except LookupError:
    print("NLTK stopwords not found. Downloading...")
    nltk.download("stopwords")
    stop_words_list = stopwords.words("english")
# print(f"Using {len(stop_words_list)} English stop words.")

# --- 2. Load the IMDB Data ---
# Note: This assumes the 'aclImdb' folder is in your current working directory
print("Loading IMDB dataset...")
train_review = load_files("./aclImdb/train/", encoding="utf-8")
x_train, y_train = train_review.data, train_review.target

test_review = load_files("./aclImdb/test/", encoding="utf-8")
x_test, y_test = test_review.data, test_review.target
print(f"Train data size: {len(x_train)} samples")
print(f"Test data size: {len(x_test)} samples")

# --- 3. Initialize TfidfVectorizer ---
# Key Settings:
# 1. max_features=5000: Limits vocabulary size.
# 2. stop_words=stop_words_list: Uses the NLTK English stop words.
# 3. token_pattern=r"(?u)\b\w+\b": This is the pattern from the assignment's BoW example, 
#                                  ensuring single-letter words like 'a' are included.
# 4. norm=None: As an exercise, we set normalization to None (per scikit-learn note)
#               to get the raw TF * IDF values before L2-normalization.
vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words=stop_words_list,
    token_pattern=r"(?u)\b\w+\b",
    norm=None # To satisfy the note on L2 normalization
)

# --- 4. Vectorize the Training and Test Data ---

# Fit the vectorizer on the TRAINING data ONLY to learn the vocabulary and IDF values
print("Fitting TfidfVectorizer on training data...")
vectorizer.fit(x_train)

# Transform both training and testing data using the fitted vectorizer
print("Transforming data...")
X_train_tfidf = vectorizer.transform(x_train)
X_test_tfidf = vectorizer.transform(x_test)

# --- Verification and Output ---
print("\n--- TF-IDF Vectorization Complete ---")
print(f"Final feature space dimension (vocabulary size): {X_train_tfidf.shape[1]}")
print(f"Shape of TF-IDF matrix (Training): {X_train_tfidf.shape}")
print(f"Shape of TF-IDF matrix (Testing): {X_test_tfidf.shape}")

# Optional: Display a few IDF values to confirm the calculation
# The vocabulary will be sorted by feature name alphabetically (default behavior)
idf_values = vectorizer.idf_
feature_names = np.array(vectorizer.get_feature_names_out())

# Find the index of a common word like 'bad' and a rare word
bad_index = np.where(feature_names == 'bad')[0][0]
print(f"\nIDF value for 'bad': {idf_values[bad_index]:.4f}")

# Find a word with a very high IDF (rare)
# The word 's' (from I's, it's, etc.) is often high due to cleaning, 
# or we can look at the last word in the vocab.
rare_word_index = -1 
print(f"IDF value for '{feature_names[rare_word_index]}': {idf_values[rare_word_index]:.4f}")

NLTK stopwords not found. Downloading...


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\lelin\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


Loading IMDB dataset...


In [ ]:
#Problem 3] Learning with TF-IDF

In [ ]:
#Train and Predict

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# --- Initialize and Train the Model ---
# Logistic Regression is a simple yet powerful linear model for binary classification.
print("Training Logistic Regression model...")
model = LogisticRegression(solver='liblinear', random_state=42)
model.fit(X_train_tfidf, y_train)

# --- Predict on the Test Set ---
y_pred = model.predict(X_test_tfidf)

# --- Evaluate the Model ---
accuracy = accuracy_score(y_test, y_pred)

print("\n--- Model Evaluation ---")
print(f"Test Accuracy: {accuracy:.4f}")
print("Classification complete.")

In [ ]:
#2. Testing the Impact of Hyperparameters

In [ ]:
# --- Problem 2 (Modified for Bigrams) ---
vectorizer_bigram = TfidfVectorizer(
    max_features=5000,
    stop_words=stop_words_list,
    token_pattern=r"(?u)\b\w+\b",
    ngram_range=(1, 2),  # <--- The change!
    norm=None
)

# ... (Rest of Problem 2: fit, transform) ...
vectorizer_bigram.fit(x_train)
X_train_bigram_tfidf = vectorizer_bigram.transform(x_train)
X_test_bigram_tfidf = vectorizer_bigram.transform(x_test)

# --- Problem 3 (Retrain) ---
# Retrain the model using the new vectors
model_bigram = LogisticRegression(solver='liblinear', random_state=42)
model_bigram.fit(X_train_bigram_tfidf, y_train)
y_pred_bigram = model_bigram.predict(X_test_bigram_tfidf)

accuracy_bigram = accuracy_score(y_test, y_pred_bigram)
print(f"Test Accuracy with Bigrams: {accuracy_bigram:.4f}")

In [ ]:
#Problem 4: Scratch mounting of TF-IDF

In [ ]:
import numpy as np
import math
from collections import Counter

mini_dataset = [
    "This movie is SOOOO funny!!!",
    "What a movie! I never",
    "best movie ever!!!!! this movie",
]

def preprocess_and_tokenize(text):
    """Clean and tokenize the text."""
    # Simplified cleaning: lower, keep only letters and spaces, then split
    text = text.lower()
    text = ''.join(c for c in text if c.isalnum() or c.isspace()).strip()
    return text.split()

tokenized_corpus = [preprocess_and_tokenize(d) for d in mini_dataset]
N = len(tokenized_corpus)

# 1. Build Global Vocabulary and Document Frequency (df)
vocab = set()
df = Counter()
doc_lengths = [] 

for doc in tokenized_corpus:
    vocab.update(doc)
    # Count unique words in this document for df
    df.update(set(doc))
    # Calculate total word count for the Standard TF denominator
    doc_lengths.append(len(doc))

vocab = sorted(list(vocab)) # Use sorted list for consistent column order
vocab_size = len(vocab)
print(f"Vocabulary Size: {vocab_size}")
print(f"Document Lengths: {doc_lengths}\n")

# --- TF-IDF Matrix Initialization ---
tfidf_standard = np.zeros((N, vocab_size))
tfidf_sklearn = np.zeros((N, vocab_size))

# --- Function Definitions for Clarity ---

def calculate_standard_idf(t_name, N, df_count):
    """IDF = log(N / df(t))"""
    return math.log(N / df_count[t_name])

def calculate_sklearn_idf(t_name, N, df_count):
    """IDF = log((1 + N) / (1 + df(t))) + 1"""
    return math.log((1 + N) / (1 + df_count[t_name])) + 1

# --- Main Calculation Loop ---
for doc_idx, doc in enumerate(tokenized_corpus):
    tf_raw = Counter(doc)
    length = doc_lengths[doc_idx] # Total words in this document
    
    for term_idx, term_name in enumerate(vocab):
        # Raw Term Frequency
        ntd = tf_raw.get(term_name, 0)
        if ntd == 0:
            continue

        # --- Standard Formula Calculation ---
        # 1. TF (Normalized)
        tf_std = ntd / length
        # 2. IDF (Standard)
        idf_std = calculate_standard_idf(term_name, N, df)
        # 3. TF-IDF
        tfidf_standard[doc_idx, term_idx] = tf_std * idf_std

        # --- Scikit-learn Formula Calculation ---
        # 1. TF (Raw Count)
        tf_skl = ntd
        # 2. IDF (Scikit-learn formula)
        idf_skl = calculate_sklearn_idf(term_name, N, df)
        # 3. TF-IDF
        tfidf_sklearn[doc_idx, term_idx] = tf_skl * idf_skl

# --- Display Results ---
print("--- Standard TF-IDF Matrix (TF * log(N/df)) ---")
df_std = pd.DataFrame(tfidf_standard, index=[f"Doc {i+1}" for i in range(N)], columns=vocab)
display(df_std.round(4))

print("\n--- Scikit-learn TF-IDF Matrix (Raw TF * (log((1+N)/(1+df)) + 1)) ---")
df_skl = pd.DataFrame(tfidf_sklearn, index=[f"Doc {i+1}" for i in range(N)], columns=vocab)
display(df_skl.round(4))

In [ ]:
#Problem 5] Pre-processing of corpus

In [ ]:
import re
from sklearn.datasets import load_files # Assuming you still need to load the data if you restart the notebook

# --- Re-Load Data (Safety Check, run only if x_train/x_test are undefined) ---
try:
    # Check if data is already loaded from Problem 2
    x_train[0]
except NameError:
    # Load data if variables are not in memory
    print("Loading IMDB dataset...")
    train_review = load_files("./aclImdb/train/", encoding="utf-8")
    x_train, y_train = train_review.data, train_review.target
    test_review = load_files("./aclImdb/test/", encoding="utf-8")
    x_test, y_test = test_review.data, test_review.target

# --- The Preprocessing Function ---

def preprocess_text(text):
    """
    Cleans a single string of text: lowercasing, removing URLs, 
    removing punctuation/special characters, and tokenizing.
    
    Args:
        text (str): The raw review text.
        
    Returns:
        list: A list of cleaned words (tokens).
    """
    # 1. Lowercasing
    text = text.lower()
    
    # 2. URL Removal
    # This regex pattern finds common http/https links
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    text = url_pattern.sub(r'', text)
    
    # 3. Special Character/Punctuation Removal
    # This regex keeps only letters (a-z), numbers (0-9), and spaces.
    # It removes !@#$%^&*()_+={}|[]\:";'<>?,./ and other symbols.
    text = re.sub(r'[^a-z0-9\s]', '', text) 
    
    # 4. Tokenization (Splitting words into a list)
    # Also removes extra spaces that might have been created by cleaning
    tokens = text.split()
    
    return tokens

# --- Apply Preprocessing to the Entire Corpus ---

print("Applying preprocessing to training corpus...")
# Note: This operation can take a few seconds due to the size of the dataset (25,000 reviews).
X_train_processed = [preprocess_text(review) for review in x_train]

print("Applying preprocessing to test corpus...")
X_test_processed = [preprocess_text(review) for review in x_test]

# --- Verification ---

print("\nPreprocessing complete.")
print(f"Original Text (Sample 0):\n{x_train[0][:200]}...") # Show original start

print("\nProcessed Tokens (Sample 0):")
# The output is a list of lists of strings, which is the format required for Word2Vec (Problem 6)
print(X_train_processed[0][:30]) 
print(f"\nNumber of training sentences (lists): {len(X_train_processed)}")

In [ ]:
#Problem 6: Learning Word2Vec

In [ ]:
# Prepare the Corpus

In [ ]:
from gensim.models import Word2Vec

# Combine the processed training and test reviews into one corpus
# X_train_processed and X_test_processed are lists of lists of strings
full_corpus = X_train_processed + X_test_processed 
print(f"Total number of documents/sentences in corpus: {len(full_corpus)}")

# --- Word2Vec Parameters ---
# We will use common, reasonable parameters:
# 1. size (vector dimension): 100 is standard (your example used 10)
# 2. window: 5 (size of context window)
# 3. min_count: 5 (ignore all words with total frequency less than this)
# 4. workers: Use all available cores for faster training

vector_size = 100
window_size = 5
min_word_count = 5 # Filter out rare words for better quality embeddings
sg = 0 # 0=CBOW (default), 1=Skip-gram. We'll stick to CBOW for simplicity.
epochs = 10 # Number of iterations over the corpus

# --- Initialize the Model ---
print("Initializing Word2Vec model...")
word2vec_model = Word2Vec(
    vector_size=vector_size,
    window=window_size,
    min_count=min_word_count,
    sg=sg,
    workers=4, # Use 4 cores (adjust based on your machine)
    seed=42 # for reproducibility
)

# --- Build Vocabulary ---
# Scans the data to initialize the internal model tables (builds the vocabulary)
print("Building vocabulary...")
word2vec_model.build_vocab(full_corpus)
corpus_count = word2vec_model.corpus_count
initial_vocab_size = len(word2vec_model.wv)
print(f"Initial Vocabulary Size (min_count={min_word_count}): {initial_vocab_size}")

# --- Train the Model ---
# This is the learning step where the neural network adjusts the weights
print(f"Starting training for {epochs} epochs...")
# Note: total_examples is required when passing the raw corpus to train()
word2vec_model.train(
    full_corpus, 
    total_examples=corpus_count, 
    epochs=epochs
)
print("Training complete.")
print(f"Final Vocabulary Size (Learned words): {len(word2vec_model.wv)}")

In [ ]:
#Verification and Exploration

In [ ]:
# 1. Access the vector for a specific word
word = "movie"
if word in word2vec_model.wv:
    print(f"\nVector for '{word}' (first 5 dimensions):")
    print(word2vec_model.wv[word][:5].round(4))
else:
    print(f"\n'{word}' was filtered out (count < {min_word_count}).")


# 2. Find similar words (Semantic relationships)
# This is the key benefit of Word2Vec over BoW/TF-IDF
positive_word = "good"
if positive_word in word2vec_model.wv:
    print(f"\nTop 5 words most similar to '{positive_word}':")
    # model.wv.most_similar returns a list of (word, similarity_score) tuples
    similar_words = word2vec_model.wv.most_similar(positive=positive_word, topn=5)
    for w, score in similar_words:
        print(f"  {w}: {score:.4f}")
else:
    print(f"\n'{positive_word}' was filtered out (count < {min_word_count}).")

In [ ]:
# [Problem 7] (Advance assignment) Vector Visualization

In [ ]:
## Word Similarity Exploration (wv.most_similar)

In [ ]:
print("--- Word Similarity Exploration ---")

# Select a few words to find similar terms for
test_words = ["terrible", "oscar", "plot", "actor"]

for word in test_words:
    if word in word2vec_model.wv:
        print(f"\nTop 5 words most similar to '{word.upper()}':")
        # model.wv.most_similar returns a list of (word, similarity_score) tuples
        similar_words = word2vec_model.wv.most_similar(positive=word, topn=5)
        for w, score in similar_words:
            print(f"  {w:<10}: {score:.4f}")
    else:
        print(f"\n'{word}' not found in vocabulary (min_count filter).")

In [ ]:
# [Issue 8] (Advance assignment) Classification of movie reviews using Word2Vec

In [ ]:
##  Vector Averaging Function

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

def document_vector(word2vec_model, doc):
    """
    Creates a single vector for a document by averaging the vectors 
    of all words present in the model's vocabulary.
    
    Args:
        word2vec_model (gensim.models.Word2Vec): The trained model.
        doc (list): A list of tokens representing the document.
        
    Returns:
        numpy.ndarray: The averaged document vector.
    """
    # Filter out words that are not in the model's vocabulary
    words = [w for w in doc if w in word2vec_model.wv]
    
    if not words:
        # If no words in the document are in the vocab, return a zero vector
        # The length should match the vector_size used in Problem 6 (e.g., 100)
        return np.zeros(word2vec_model.wv.vector_size)
    
    # Get the vectors for all remaining words
    vectors = [word2vec_model.wv[w] for w in words]
    
    # Calculate the mean (average) of all the vectors
    return np.mean(vectors, axis=0)

print("Document vector averaging function defined.")

In [ ]:
## Apply Averaging to Corpus

In [ ]:
print("Converting training reviews to averaged vectors...")
# Use a list comprehension to apply the function to every review
X_train_w2v = np.array([document_vector(word2vec_model, doc) for doc in X_train_processed])

print("Converting test reviews to averaged vectors...")
X_test_w2v = np.array([document_vector(word2vec_model, doc) for doc in X_test_processed])

print("\nWord2Vec Feature Matrix created.")
print(f"Shape of Word2Vec Training features: {X_train_w2v.shape}")
print(f"Shape of Word2Vec Testing features: {X_test_w2v.shape}")

In [ ]:
##  Classification

In [ ]:
# --- Initialize and Train the Model ---
# We use the same classifier as Problem 3
print("Training Logistic Regression model on Word2Vec features...")
w2v_model = LogisticRegression(solver='liblinear', random_state=42, max_iter=1000)
w2v_model.fit(X_train_w2v, y_train)

# --- Predict and Evaluate ---
y_pred_w2v = w2v_model.predict(X_test_w2v)
accuracy_w2v = accuracy_score(y_test, y_pred_w2v)

print("\n--- Model Evaluation (Word2Vec) ---")
print(f"Test Accuracy (Word2Vec Averaging): {accuracy_w2v:.4f}")